<a href="https://colab.research.google.com/github/kauazin05/CIAO_KKG_2026/blob/main/lab1_aula_10_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install scikit-fuzzy -q
!mkdir -p src

In [2]:
%%writefile src/model.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

def criar_sistema_fuzzy():
    impacto = ctrl.Antecedent(np.arange(0, 101, 1), 'impacto')
    explorabilidade = ctrl.Antecedent(np.arange(0, 11, 1), 'explorabilidade')
    urgencia = ctrl.Consequent(np.arange(0, 101, 1), 'urgencia')

    urgencia.defuzzify_method = 'centroid'

    impacto['baixo'] = fuzzy.trimf(impacto.universe, [0, 0, 40])
    impacto['medio'] = fuzzy.trimf(impacto.universe, [20, 50, 80])
    impacto['alto'] = fuzzy.trapmf(impacto.universe, [60, 80, 100, 100])

    explorabilidade['baixa'] = fuzzy.trimf(explorabilidade.universe, [0, 0, 4])
    explorabilidade['moderada'] = fuzzy.trimf(explorabilidade.universe, [2, 5, 8])
    explorabilidade['alta'] = fuzzy.trimf(explorabilidade.universe, [6, 10, 10])

    urgencia['baixa'] = fuzzy.trimf(urgencia.universe, [0, 0, 30])
    urgencia['media'] = fuzzy.trimf(urgencia.universe, [20, 50, 70])
    urgencia['alta'] = fuzzy.trimf(urgencia.universe, [60, 80, 90])
    urgencia['critica'] = fuzzy.trapmf(urgencia.universe, [80, 90, 100, 100])

    return impacto, explorabilidade, urgencia

Writing src/model.py


In [3]:
%%writefile engine.py
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from skfuzzy import control as ctrl
from src.model import criar_sistema_fuzzy


def construir_motor_inferencia():
    impacto, explorabilidade, urgencia = criar_sistema_fuzzy()

    regra1 = ctrl.Rule(impacto['baixo'] & explorabilidade['baixa'], urgencia['baixa'])
    regra2 = ctrl.Rule(impacto['baixo'] & explorabilidade['moderada'], urgencia['baixa'])
    regra3 = ctrl.Rule(impacto['baixo'] & explorabilidade['alta'], urgencia['media'])
    regra4 = ctrl.Rule(impacto['medio'] & explorabilidade['baixa'], urgencia['media'])
    regra5 = ctrl.Rule(impacto['medio'] & explorabilidade['moderada'], urgencia['alta'])
    regra6 = ctrl.Rule(impacto['medio'] & explorabilidade['alta'], urgencia['alta'])
    regra7 = ctrl.Rule(impacto['alto'] & explorabilidade['baixa'], urgencia['critica'])
    regra8 = ctrl.Rule(impacto['alto'] & explorabilidade['moderada'], urgencia['critica'])
    regra9 = ctrl.Rule(impacto['alto'] & explorabilidade['alta'], urgencia['critica'])

    sistema_controle = ctrl.ControlSystem([
        regra1, regra2, regra3,
        regra4, regra5, regra6,
        regra7, regra8, regra9
    ])
    return ctrl.ControlSystemSimulation(sistema_controle)


def gerar_superficie_3d(simulador, nome_arquivo='superficie_decisao.png'):
    Path("results").mkdir(exist_ok=True)
    x, y = np.meshgrid(np.linspace(0, 100, 20), np.linspace(0, 10, 20))
    z = np.zeros_like(x)

    for i in range(20):
        for j in range(20):
            simulador.input['impacto'] = x[i, j]
            simulador.input['explorabilidade'] = y[i, j]
            simulador.compute()
            z[i, j] = simulador.output['urgencia']

    fig = plt.figure(figsize=(9, 6))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(x, y, z, cmap='viridis', edgecolor='none', alpha=0.9)
    ax.set_xlabel('Impacto no Negócio (%)')
    ax.set_ylabel('Explorabilidade (0-10)')
    ax.set_zlabel('Urgência Calculada (%)')
    plt.title('Superfície de Decisão Fuzzy — SOC Autônomo')
    fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
    plt.tight_layout()
    plt.savefig(f'results/{nome_arquivo}')
    plt.close()
    print(f"Sucesso: Imagem 'results/{nome_arquivo}' gerada com êxito!")


def simular_casos_teste(simulador):
    casos = [
        ("Mínimo Absoluto", 0.0, 0.0),
        ("Baixo Impacto + Alta Explorabilidade (Transição)", 10.0, 9.0),
        ("Médio Impacto + Moderado", 50.0, 5.0),
        ("Máximo Absoluto", 100.0, 10.0)
    ]
    print("\n=== SIMULAÇÕES DE TESTE DE INFERÊNCIA ===")
    for nome, imp, exp in casos:
        simulador.input['impacto'] = imp
        simulador.input['explorabilidade'] = exp
        simulador.compute()
        res = simulador.output['urgencia']
        print(f"[{nome}] -> Imp: {imp}% | Exp: {exp} => Urgência: {res:.2f}%")


if __name__ == "__main__":
    sim = construir_motor_inferencia()
    simular_casos_teste(sim)
    gerar_superficie_3d(sim)

Writing engine.py


In [4]:
!python engine.py


=== SIMULAÇÕES DE TESTE DE INFERÊNCIA ===
[Mínimo Absoluto] -> Imp: 0.0% | Exp: 0.0 => Urgência: 10.00%
[Baixo Impacto + Alta Explorabilidade (Transição)] -> Imp: 10.0% | Exp: 9.0 => Urgência: 46.50%
[Médio Impacto + Moderado] -> Imp: 50.0% | Exp: 5.0 => Urgência: 76.67%
[Máximo Absoluto] -> Imp: 100.0% | Exp: 10.0 => Urgência: 92.22%
Sucesso: Imagem 'results/superficie_decisao.png' gerada com êxito!
